In [47]:
import numpy as np
import pandas as pdb
import matplotlib.pyplot as plt

In [48]:

class Value:
    """ stores a single scalar value and its gradient """

    def __init__(self, data, _children=()):
        self.data = data
        self.grad = 0
        
        self._backward = lambda: None
        self._prev = set(_children)        

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers"
        out = Value(self.data**other, (self,))

        def _backward():
            self.grad += (other * self.data**(other-1)) * out.grad
        out._backward = _backward

        return out

    def relu(self):
        out = Value(0 if self.data < 0 else self.data, (self,))

        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward

        return out

    def tanh(self):
        out = Value(np.tanh(self.data), (self,))

        def _backward():
            self.grad += (1-out.data**2) * out.grad
        out._backward = _backward

        return out

    def backward(self):

        # topological order all of the children in the graph
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        # go one variable at a time and apply the chain rule to get its gradient
        self.grad = 1
        for v in reversed(topo):
            v._backward()

    def __neg__(self): # -self
        return self * -1

    def __radd__(self, other): # other + self
        return self + other

    def __sub__(self, other): # self - other
        return self + (-other)

    def __rsub__(self, other): # other - self
        return other + (-self)

    def __rmul__(self, other): # other * self
        return self * other

    def __truediv__(self, other): # self / other
        return self * other**-1

    def __rtruediv__(self, other): # other / self
        return other * self**-1

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"


In [49]:
import random
class Neuron:

    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(0)        

    def __call__(self, x):
        act = sum((wi*xi for wi,xi in zip(self.w, x)), self.b)
        return act.tanh()

    def parameters(self):
        return self.w + [self.b]

    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0
    
    def __repr__(self):
        return f"Linear Neuron({len(self.w)})"

class Layer:

    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0
    
    def __repr__(self):
        return f"Layer"

class MLP:

    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0
    
    def __repr__(self):
        return f"MLP"

In [50]:
mlp = MLP(3,[4,4,1])
etha = 0.05
print(len(mlp.parameters()))

41


In [51]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
    ]
ys = [1.0, -1.0, -1.0, 1.0]

In [56]:
for k in range(1000):

    # Forward pass
    ypred = [mlp(x) for x in xs]    
    loss = sum((yout-ygt)**2 for ygt, yout in zip(ys, ypred))/len(ypred)

    # Bacward pass
    mlp.zero_grad()
    loss.backward()

    # Update
    for p in mlp.parameters():
        p.data += -etha*p.grad

    if k%10 == 0:print(k, loss.data)
    

0 0.0001232444688301744
10 0.00012291635495817176
20 0.00012258991863288158
30 0.00012226514718787315
40 0.0001219420280832043
50 0.00012162054890384488
60 0.00012130069735813258
70 0.0001209824612762514
80 0.00012066582860872392
90 0.0001203507874249354
100 0.00012003732591168008
110 0.00011972543237171463
120 0.00011941509522235109
130 0.00011910630299406034
140 0.00011879904432909362
150 0.00011849330798013625
160 0.00011818908280896134
170 0.00011788635778512423
180 0.00011758512198466254
190 0.00011728536458881351
200 0.0001169870748827616
210 0.00011669024225439268
220 0.00011639485619307172
230 0.0001161009062884328
240 0.00011580838222919609
250 0.00011551727380198862
260 0.00011522757089019245
270 0.0001149392634728042
280 0.00011465234162331022
290 0.00011436679550857988
300 0.00011408261538777578
310 0.00011379979161127071
320 0.00011351831461959139
330 0.00011323817494236665
340 0.00011295936319729982
350 0.00011268187008914438
360 0.00011240568640870672
370 0.0001121308030